# Water Body Segmentation & Flood Mapping
**PyGeoVision v2.0 — Notebook 07**

## Real-World Problem
Emergency response teams need rapid flood extent mapping after extreme rainfall.
Sentinel-2 optical + NDWI + GeoAI segmentation provides maps within hours.

```bash
pip install "pygeovision[geo,train]"
```

In [ ]:
import pygeovision as pgv
import numpy as np, rasterio, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/07_water_flood")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BBOX = (14.35, 49.95, 14.75, 50.25)   # Prague, Czech Republic — Sept 2024 flood
client = pgv.PyGeoVision()
print("Study area: Prague & Bohemian basin — September 2024 flood event")

## Step 1: Pre/Post-Flood Imagery

In [ ]:
def get_scene(date_range, label, bbox=BBOX):
    results = client.search(
        bbox=bbox, date_range=date_range,
        providers=["planetary_computer"], cloud_cover_max=30,
        sort_by="cloud_cover", limit=3,
    )
    print(f"  {label}: {len(results)} scenes")
    if results:
        dl = client.download(results[:1], output_dir=BASE_DIR/label.replace(" ","_"),
                              post_process=["reproject:EPSG:32633","cog"])
        if dl and dl[0].success:
            return dl[0].path
    return None

pre_path  = get_scene(("2024-08-01","2024-09-10"),  "pre_flood")
post_path = get_scene(("2024-09-14","2024-09-22"), "post_flood")
print(f"\nPre-flood : {pre_path or 'not available'}")
print(f"Post-flood: {post_path or 'not available'}")

## Step 2: NDWI Water Detection

In [ ]:
import rasterio

def compute_ndwi(scene_path, thr=0.0):
    with rasterio.open(scene_path) as src:
        bands = src.read().astype(np.float32) / 10000.0
        # S2: B02=0(blue), B03=1(green), B04=2(red), B08=3(nir)
        green = bands[min(1, len(bands)-1)]
        nir   = bands[min(3, len(bands)-1)]
        denom = green + nir
        ndwi  = np.where(denom > 1e-4, (green - nir) / (denom + 1e-6), 0)
    return ndwi, (ndwi > thr).astype(np.uint8)

# Generate demo NDWI maps
np.random.seed(55)
H, W  = 256, 256
base  = np.random.randn(H,W)*0.08 - 0.15   # Mostly land

# Pre: some rivers
pre_ndwi = base.copy()
pre_ndwi[100:110, :] += 0.5   # Main river channel
pre_ndwi[120:125, 80:130] += 0.4

# Post: flooded
post_ndwi = base.copy()
post_ndwi[95:120, :] += 0.6   # Wider flood
post_ndwi[115:135, 60:180] += 0.5
post_ndwi += np.random.randn(H,W)*0.03

if pre_path and Path(pre_path).exists():
    pre_ndwi, _ = compute_ndwi(pre_path, 0.1)
if post_path and Path(post_path).exists():
    post_ndwi, _ = compute_ndwi(post_path, 0.0)

water_pre  = (pre_ndwi  > 0.1).astype(np.uint8)
water_post = (post_ndwi > 0.0).astype(np.uint8)
flood_ext  = np.maximum(water_post - water_pre, 0)

PIXEL_M = 10
pre_km2   = water_pre.sum()  * PIXEL_M**2 / 1e6
post_km2  = water_post.sum() * PIXEL_M**2 / 1e6
flood_km2 = flood_ext.sum()  * PIXEL_M**2 / 1e6

print(f"Water coverage:")
print(f"  Pre-flood  : {pre_km2:.2f} km2  ({water_pre.mean()*100:.1f}%)")
print(f"  Post-flood : {post_km2:.2f} km2  ({water_post.mean()*100:.1f}%)")
print(f"  Flood extent (new): {flood_km2:.2f} km2")

## Step 3: GeoAI Water Segmentation

In [ ]:
print("Water Segmentation Methods in PyGeoVision:")
print()
print("1. NDWI threshold (this notebook — fast, no model needed)")
print("   ndwi > 0.1  -> binary water mask")
print()
print("2. GeoAI semantic segmentation:")
print("   result = client.geoai.segment.water(scene, band_order='sentinel2')")
print()
print("3. SAM zero-shot:")
print("   masks = client.geoai.sam.generate_masks(scene, points_per_side=32)")
print("   water = [m for m in masks if m['water_probability'] > 0.8]")
print()
print("4. Prithvi-EO-2.0 flood detection:")
print("   tasks = PrithviTasks('prithvi_eo_2_0')")
print("   flood = tasks.flood_detection(post_scene)")
print()
print("5. Dynamic World labels (Google, near real-time):")
print("   dw = client.labeling.dynamic_world(bbox,")
print("            date_range=['2024-09-14','2024-09-20'])")
print()

if post_path and Path(post_path).exists():
    result = client.geoai.segment.water(post_path, band_order="sentinel2",
                                          output_path=str(BASE_DIR/"water_geoai.tif"))
    print(f"GeoAI result: {result}")

## Step 4: Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(pre_ndwi,  cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[0].set_title("NDWI Pre-Flood\n(Blue = water)", fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(post_ndwi, cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[1].set_title("NDWI Post-Flood\n(Blue = water)", fontsize=11, fontweight='bold')
axes[1].axis('off')

flood_rgb = np.zeros((*flood_ext.shape, 3))
flood_rgb[water_pre > 0]  = [0.1, 0.3, 0.8]   # Permanent water (blue)
flood_rgb[flood_ext > 0]  = [0.9, 0.1, 0.1]   # New flood (red)
axes[2].imshow(flood_rgb)
axes[2].set_title(f"Flood Extent Map\nBlue=permanent  Red=flooded ({flood_km2:.1f} km2)",
                   fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.suptitle("Prague Flood — September 2024 | NDWI Analysis",
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"flood_map.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Flood extent: {flood_km2:.2f} km2  ({flood_ext.mean()*100:.1f}% of scene)")

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 07 — Water Body Segmentation & Flood Mapping")
print("=" * 60)
print(f"Study area    : Prague, Czech Republic")
print(f"Event         : September 2024 flood")
print(f"Flood extent  : {flood_km2:.2f} km2 (new inundation)")
print()
print("Methods:")
print("  NDWI > 0.1     : fast operational flood mapping")
print("  GeoAI segment  : accurate boundary delineation")
print("  SAR Sentinel-1 : cloud-independent backup")
print()
print("Next: 08_solar_panel_detection_energy_assessment.ipynb")